# Step 1: 데이터 준비

딥페이크 탐지 실습을 위한 데이터를 준비합니다.

## 실습 목표
- 딥페이크 탐지용 데이터셋 구조 이해
- 사전 준비된 데이터 다운로드
- S3 버킷에 데이터 업로드

## 1.1 환경 설정

In [ ]:
import os
import boto3
import sagemaker
from pathlib import Path

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path(os.getcwd()).parent
print(f"Project Root: {PROJECT_ROOT}")

# SageMaker 세션 설정
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

# S3 버킷 설정
bucket = sagemaker_session.default_bucket()
prefix = 'deepfake-detection'

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

## 1.2 Kaggle API 설정

Kaggle의 140k Real and Fake Faces 데이터셋을 사용합니다.

**데이터셋 정보:**
- 출처: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
- 크기: 약 4GB (140,000장 이미지)
- 구성: Real 70,000장 + Fake 70,000장
- 장점: 신청 없이 바로 다운로드 가능

**사전 준비:**
1. [Kaggle](https://www.kaggle.com) 계정 생성
2. Account Settings → API → "Create New Token" 클릭
3. 다운로드된 `kaggle.json` 파일 확인

In [ ]:
# Kaggle 패키지 설치
!pip install -q kaggle

# Kaggle API 설정
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

# ============================================
# ⚠️ kaggle.json 파일 업로드 필요!
# ============================================
# 방법 1: JupyterLab 파일 브라우저에서 kaggle.json을 ~/.kaggle/ 폴더에 업로드
# 방법 2: 아래 코드 실행 후 직접 입력

print("Kaggle API 설정 방법:")
print("1. Kaggle.com → Account → API → Create New Token")
print("2. 다운로드된 kaggle.json을 ~/.kaggle/ 폴더에 업로드")
print("3. 또는 아래 셀에서 직접 입력")

In [ ]:
# (선택) Kaggle 자격 증명 직접 입력
# kaggle.json 파일이 없는 경우에만 실행하세요

import json
from pathlib import Path

kaggle_path = Path.home() / '.kaggle' / 'kaggle.json'

if not kaggle_path.exists():
    print("Kaggle 자격 증명을 입력하세요:")
    username = input("Kaggle Username: ")
    api_key = input("Kaggle API Key: ")
    
    credentials = {"username": username, "key": api_key}
    
    with open(kaggle_path, 'w') as f:
        json.dump(credentials, f)
    
    os.chmod(kaggle_path, 0o600)
    print(f"✅ Kaggle 자격 증명 저장 완료: {kaggle_path}")
else:
    print(f"✅ Kaggle 자격 증명 확인됨: {kaggle_path}")

In [ ]:
# 데이터셋 다운로드 (약 4GB, 5-10분 소요)
print("140k Real and Fake Faces 데이터셋 다운로드 중...")
print("(약 4GB, 인터넷 속도에 따라 5-10분 소요)\n")

!kaggle datasets download -d xhlulu/140k-real-and-fake-faces --unzip -p ./kaggle_data

print("\n✅ 다운로드 완료!")

## 1.3 데이터 구조 변환

Kaggle 데이터셋을 Workshop 형식에 맞게 변환합니다.

**원본 구조 (Kaggle):**
```
real_vs_fake/
├── real_vs_fake/
│   ├── train/
│   │   ├── real/
│   │   └── fake/
│   └── valid/
│       ├── real/
│       └── fake/
```

**변환 후 구조:**
```
data/
├── train/    # 학습 데이터
├── val/      # 검증 데이터
└── test/     # 테스트 데이터 (Before/After 비교용)
```

In [ ]:
import shutil
import random
from pathlib import Path

# 원본 Kaggle 데이터 경로
kaggle_dir = Path('./kaggle_data/real_vs_fake/real_vs_fake')

# 목표 데이터 디렉토리
data_dir = Path('./data')

# 디렉토리 초기화
if data_dir.exists():
    shutil.rmtree(data_dir)

# train/val/test 구조 생성
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        (data_dir / split / label).mkdir(parents=True, exist_ok=True)

print("데이터 구조 변환 중...")

# 워크샵용 샘플 크기 설정 (전체 140k는 너무 큼)
TRAIN_SIZE = 1000  # 클래스당 1000장
VAL_SIZE = 200     # 클래스당 200장
TEST_SIZE = 200    # 클래스당 200장

for label in ['real', 'fake']:
    # Kaggle train 데이터에서 샘플링
    src_train = kaggle_dir / 'train' / label
    all_images = list(src_train.glob('*.jpg'))
    random.shuffle(all_images)
    
    # Train 데이터 복사
    for img in all_images[:TRAIN_SIZE]:
        shutil.copy(img, data_dir / 'train' / label / img.name)
    
    # Val 데이터 복사 (Kaggle valid 폴더에서)
    src_val = kaggle_dir / 'valid' / label
    val_images = list(src_val.glob('*.jpg'))[:VAL_SIZE]
    for img in val_images:
        shutil.copy(img, data_dir / 'val' / label / img.name)
    
    # Test 데이터 복사 (train에서 추가 샘플링)
    test_images = all_images[TRAIN_SIZE:TRAIN_SIZE+TEST_SIZE]
    for img in test_images:
        shutil.copy(img, data_dir / 'test' / label / img.name)
    
    print(f"{label}: train={TRAIN_SIZE}, val={len(val_images)}, test={len(test_images)}")

print("\n✅ 데이터 변환 완료!")

In [ ]:
# 데이터 구조 확인
print("데이터 디렉토리 구조:")
!find ./data -type d

print("\n데이터 개수 확인:")
total = 0
for split in ['train', 'val', 'test']:
    split_dir = data_dir / split
    if split_dir.exists():
        real_count = len(list((split_dir / 'real').glob('*')))
        fake_count = len(list((split_dir / 'fake').glob('*')))
        print(f"{split}: Real={real_count}, Fake={fake_count}, Total={real_count+fake_count}")
        total += real_count + fake_count

print(f"\n총 이미지 수: {total}장")

## 1.4 샘플 이미지 확인

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, label in enumerate(['real', 'fake']):
    label_dir = data_dir / 'test' / label
    if label_dir.exists():
        images = list(label_dir.glob('*.jpg')) + list(label_dir.glob('*.png'))
        samples = random.sample(images, min(4, len(images)))
        
        for j, img_path in enumerate(samples):
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            axes[i, j].set_title(f"{label.upper()}")

plt.suptitle("샘플 이미지 (Real vs Fake)", fontsize=14)
plt.tight_layout()
plt.show()

## 1.5 내 S3 버킷에 업로드

다운로드한 데이터를 본인의 S3 버킷에 업로드합니다.

In [ ]:
# S3에 데이터 업로드
print(f"내 S3 버킷에 업로드 중: s3://{bucket}/{prefix}/data/")

s3_data_path = sagemaker_session.upload_data(
    path='./data',
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)

print(f"\n✅ 업로드 완료: {s3_data_path}")

## 1.6 설정 저장

다음 노트북에서 사용할 설정을 저장합니다.

In [ ]:
import json

config = {
    'bucket': bucket,
    'prefix': prefix,
    'project_root': str(PROJECT_ROOT),
    's3_data_path': s3_data_path,
    's3_train_path': f's3://{bucket}/{prefix}/data/train',
    's3_val_path': f's3://{bucket}/{prefix}/data/val',
    's3_test_path': f's3://{bucket}/{prefix}/data/test',
    'local_test_path': str(data_dir.absolute() / 'test'),
    'role': role,
    'region': region
}

# 프로젝트 루트에 config.json 저장
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ 설정 저장 완료: {config_path}")
print("\n저장된 설정:")
print(json.dumps(config, indent=2))

## ✅ 완료!

데이터 준비가 완료되었습니다.

**확인 사항:**
- [x] 딥페이크 데이터 다운로드 완료
- [x] Train/Val/Test 데이터 확인
- [x] 내 S3 버킷에 업로드 완료
- [x] config.json 저장 완료

---

**➡️ 다음 단계: `2_before_evaluation/evaluate_before.ipynb`**

Fine-tuning 전 모델의 성능을 평가합니다.